# Fraud Detection under Concept Drift
## 1. Introduction
Financial fraud detection systems often degrade over time due to **Concept Drift**—the phenomenon where fraudsters constantly evolve their tactics to bypass defenses.

This project demonstrates a robust, production-ready machine learning pipeline for fraud detection.
Our primary objective is to evaluate multiple models in a strictly temporal (time-based) training and testing environment to simulate real-world conditions.


### Setting Reproducibility
To ensure the pipeline is reproducible and consistent, we set random seeds across the board.


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(42)

# Add src to path to allow seamless imports
import sys
sys.path.append(os.path.abspath('..'))


## 2. Data Loading
Instead of writing complex preprocessing code in the notebook, we import our dedicated `load_data` function from `src.preprocess` to keep the logic modular and clean.


In [ ]:
from src.preprocess import load_data

data_path = '../data/raw_data.csv'
raw_df = load_data(data_path)
display(raw_df.head())


## 3. Exploratory Data Analysis (EDA)
Before processing the data, let's explore the class imbalance and feature distributions.


In [ ]:
print(f"Dataset Shape: {raw_df.shape}")

target = 'fraud_flag' if 'fraud_flag' in raw_df.columns else 'status'
print("\nClass Distribution:\n", raw_df[target].value_counts(normalize=True))

plt.figure(figsize=(6, 4))
raw_df[target].value_counts().plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Fraud Distribution (Class Imbalance)')
plt.xticks(rotation=0)
plt.tight_layout()
os.makedirs('../outputs/plots', exist_ok=True)
plt.savefig('../outputs/plots/eda_fraud_distribution.png', dpi=300)
plt.show()


## 4. Feature Engineering
We now apply our `preprocess_features` function.
This step extracts cyclical time features (hour, day_of_week) from timestamps, imputes missing values, and strictly formats the dataset for temporal modeling.


In [ ]:
from src.preprocess import preprocess_features

df = preprocess_features(raw_df, time_col='timestamp', target_col='fraud_flag')
display(df.head())


## 5. Train-Test Split (Strict Temporal)
To prevent data leakage from the future into the past—a critical flaw in many ML pipelines—we use a sequential 60-20-20 split based on the chronological order of transactions.


In [ ]:
from src.preprocess import temporal_split_and_scale

X_train, X_val, X_test, y_train, y_val, y_test = temporal_split_and_scale(df, target_col='fraud_flag')


## 6. Model Training & Evaluation
We evaluate a diverse set of algorithmic approaches:
- **Linear Models**: Logistic Regression, SGDClassifier (Online Learning)
- **Tree-based**: Decision Tree, Random Forest, XGBoost
- **Distance-based**: Support Vector Machine (Sampled)

We rely on our `train_and_predict` and `evaluate_model` functions tightly integrated in `src/`.


In [ ]:
from src.train import train_and_predict
from src.evaluate import evaluate_model, print_results, save_results

models_to_train = ['logistic', 'tree', 'forest', 'xgboost', 'svm', 'sgd']
metrics_list = []
prob_dict = {}

for m in models_to_train:
    print(f"Training and evaluating {m}...")
    model, y_pred, y_prob = train_and_predict(m, X_train, y_train, X_val, y_val, X_test)
    metrics = evaluate_model(y_test, y_pred, y_prob, model_name=m)
    metrics_list.append(metrics)
    prob_dict[m] = y_prob

print_results(metrics_list)
save_results(metrics_list, filepath='../outputs/results.csv')


## 7. Visualization: Precision-Recall curves
For highly imbalanced datasets, PR-AUC is far more informative than standard ROC-AUC accuracy.
We plot the precision-recall trade-off across all evaluated models.


In [ ]:
from src.evaluate import plot_pr_curves

plot_pr_curves(y_test, prob_dict, save_path='../outputs/plots/pr_curves.png')
# The function automatically saves to outputs/plots and we can display the current active figure
plt.show()


## 8. Visualization: Model Comparison
Let's compare the F1-Score and PR-AUC performance metrics side-by-side to determine our strongest candidate.


In [ ]:
metrics_df = pd.DataFrame(metrics_list)
metrics_df.set_index('Model')[['F1-Score', 'PR-AUC']].plot(kind='bar', figsize=(10, 6))
plt.title('Overall Model Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/plots/model_comparison.png', dpi=300)
plt.show()


## 9. Final Conclusion
### Results Summary
By evaluating our models strictly through time (Concept Drift framework), we ensure our metrics are realistic.
- Tree-based ensembles (Random Forest, XGBoost) generally offer the highest PR-AUC and resilience against shifting fraud tactics.
- The temporal split reveals lower performance compared to randomized CV, reflecting true out-of-time conditions.
### Next Steps
In production, online algorithms (SGDClassifier) can incrementally train on new batches to continually mitigate drift without full retraining.
